# MCSL-069 — Question 30

**Section:** 8.2 Machine Learning, Session 8, Exercise a

**Question:** Train a Deep Q-Learning agent in a simple game.

The accompanying CSV file is a small, reproducible input dataset. Run the code cell below to see the implementation and its results.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import random
from collections import deque
import torch
from torch import nn
torch.manual_seed(69)
random.seed(69)
config = pd.read_csv("a_environment.csv").iloc[0]
positions, goal, episodes = int(config.positions), int(config.goal), int(config.episodes)
actions = 2
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(positions, 32), nn.ReLU(),
                                    nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, actions))
    def forward(self, state):
        return self.layers(state)
network = QNetwork()
optimizer = torch.optim.Adam(network.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
replay = deque(maxlen=500)
rewards = []
gamma, batch_size = 0.9, 32

def state_vector(position):
    vector = torch.zeros(positions)
    vector[position] = 1
    return vector

for episode in range(episodes):
    position, total_reward = 0, 0.0
    for step in range(12):
        state = state_vector(position)
        epsilon = max(0.05, 1 - episode / episodes)
        action = random.randrange(actions) if random.random() < epsilon else int(network(state).argmax())
        next_position = max(0, min(positions - 1, position + (1 if action == 1 else -1)))
        done = next_position == goal
        reward = 10.0 if done else -0.1
        replay.append((state, action, reward, state_vector(next_position), done))
        total_reward += reward
        position = next_position
        if len(replay) >= batch_size:
            batch = random.sample(replay, batch_size)
            states = torch.stack([x[0] for x in batch])
            actions_batch = torch.tensor([x[1] for x in batch])
            targets = torch.tensor([x[2] for x in batch], dtype=torch.float32)
            next_states = torch.stack([x[3] for x in batch])
            dones = torch.tensor([x[4] for x in batch], dtype=torch.float32)
            with torch.no_grad():
                targets += gamma * network(next_states).max(dim=1).values * (1 - dones)
            predicted = network(states).gather(1, actions_batch[:, None]).squeeze()
            loss = loss_fn(predicted, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if done:
            break
    rewards.append(total_reward)
print("Episodes:", episodes, "| average reward (last 20):", round(float(np.mean(rewards[-20:])), 2))
position = 0
route = [position]
for _ in range(positions + 2):
    action = int(network(state_vector(position)).argmax())
    position = max(0, min(positions - 1, position + (1 if action == 1 else -1)))
    route.append(position)
    if position == goal:
        break
print("Greedy learned route:", route, "| reached goal:", position == goal)


Episodes: 80 | average reward (last 20): 9.63
Greedy learned route: [0, 1, 2, 3, 4] | reached goal: True
